# 02 - Stress Tests, Metrics, Concepts and TCAV

This notebook consolidates **background stress testing, concept profile analysis and TCAV**.

Experimental objective:

Evaluate whether classifier predictions and post-hoc explanations remain stable when background evidence is perturbed while the animal identity is intended to remain semantically unchanged.

Analysis flow:

```text
background perturbations -> saliency drift metrics -> concept transitions -> TCAV
```

The notebook first quantifies pixel-level explanation instability, then maps failure cases into AwA2 semantic attributes and concept activation directions.


In [ ]:
from pathlib import Path
from IPython.display import Image, display, HTML
import csv
import pandas as pd
import subprocess
import sys

PROJECT_ROOT = Path.cwd().resolve()
for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (candidate / "src").exists() and (candidate / "scripts").exists():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("Could not locate the project root containing src/ and scripts/.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

MANIFEST = PROJECT_ROOT / "data" / "AWA2_subset_background20" / "awa2_manifest_subset.csv"
METADATA_ROOT = PROJECT_ROOT / "data" / "AWA2"
CHECKPOINT = PROJECT_ROOT / "outputs" / "checkpoints" / "best_resnet50_awa2.pt"

print("project_root:", PROJECT_ROOT)
print("manifest:", MANIFEST, "exists=", MANIFEST.exists())
print("checkpoint:", CHECKPOINT, "exists=", CHECKPOINT.exists())

RUN_METRICS = CHECKPOINT.exists()
RUN_ADVANCED_AUDIT = CHECKPOINT.exists()
RUN_CONCEPTS = False
RUN_TCAV = False


## Explainability Evaluation Protocol

The notebook evaluates explanations with two complementary criteria:

1. **Prediction robustness**  
   The predicted class and confidence should not collapse when only the background is perturbed.
2. **Explanation robustness**  
   The salient regions should remain partially stable if the model is still using the same semantic evidence.

For an original saliency map `M` and perturbed saliency map `M'`, the project computes:

```text
Top-k IoU     = |top_k(M) intersect top_k(M')| / |top_k(M) union top_k(M')|
Spearman rho = rank_correlation(flatten(M), flatten(M'))
Drift        = mean_absolute_difference(M, M')
```

These metrics operationalize explanation degradation. They do not assume that any single saliency method is ground truth.


## 1. Background Stress Test

AwA2 does not provide segmentation masks. The project therefore uses approximate masks to separate animal-like foreground from background-like pixels.

Perturbations:

- **Gaussian noise**: random noise applied to the background.
- **Color shift**: RGB channel manipulation on background pixels.
- **Background swap**: background replacement with uniform noise.

The goal is not perfect image editing. The goal is to test whether the classifier is sensitive to contextual cues that should not define the animal class.


In [ ]:
print('The raw perturbation grid is generated with the stress metrics command below.')


### How to interpret the figure

Look for cases where the animal remains visually recognizable but the predicted class changes. Those cases indicate that the model may be using background or context rather than only animal morphology.


## 2. Saliency Degradation Metrics

This section recomputes explanations on original and perturbed images and compares them quantitatively. The maintained method list includes Captum-backed attribution methods plus Score-CAM for report parity:

- **Grad-CAM** and **Score-CAM** compare two activation-level localization strategies, one gradient-weighted and one score-weighted.
- **Integrated Gradients** and **Expected Gradients** compare a single-baseline path attribution with a distributed-baseline expectation.

Metrics:

- **Prediction change rate**: fraction of examples whose predicted class changes.
- **Confidence drop**: decrease in predicted probability after perturbation.
- **Top-20% IoU**: overlap between the most salient pixels before and after perturbation.
- **Spearman correlation**: similarity of the saliency ranking across all pixels.
- **Saliency drift**: average absolute change between maps.

Low IoU or low Spearman means the explanation moved substantially. Method differences should be interpreted as differences in the attribution question, not as a direct ranking of visual quality.


In [ ]:
XAI_METHODS = ['gradcam', 'scorecam', 'integrated_gradients', 'expected_gradients']
STRESS_METRICS_CSV = PROJECT_ROOT / 'outputs' / 'reports' / 'phase5_saliency_metrics_notebook.csv'
PERTURBATION_GRID_FIGURE = PROJECT_ROOT / 'outputs' / 'figures' / 'phase5_perturbations_notebook.png'
SALIENCY_COMPARISON_FIGURE = PROJECT_ROOT / 'outputs' / 'figures' / 'phase5_saliency_comparison_notebook.png'

if RUN_METRICS:
    cmd = [
        sys.executable,
        str(PROJECT_ROOT / 'scripts' / 'experiments' / 'run_background_stress_metrics.py'),
        '--manifest', str(MANIFEST),
        '--checkpoint', str(CHECKPOINT),
        '--csv-output', str(STRESS_METRICS_CSV),
        '--perturbation-figure-output', str(PERTURBATION_GRID_FIGURE),
        '--figure-output', str(SALIENCY_COMPARISON_FIGURE),
        '--max-images', '4',
        '--ig-steps', '16',
        '--expected-gradient-samples', '12',
        '--scorecam-max-channels', '48',
        '--scorecam-batch-size', '16',
        '--top-percent', '20',
        '--mask-strategy', 'center_ellipse',
        '--foreground-scale', '0.68',
        '--xai-methods', *XAI_METHODS,
    ]
    print(' '.join(cmd))
    subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)
else:
    print('metric generation skipped; set RUN_METRICS=True to recompute all attribution methods')

print('', STRESS_METRICS_CSV)
if STRESS_METRICS_CSV.exists():
    with STRESS_METRICS_CSV.open('r', newline='', encoding='utf-8') as handle:
        rows = list(csv.DictReader(handle))
    print('rows:', len(rows))
    methods = sorted({row['xai_method'] for row in rows})
    print('xai methods:', methods)
    for row in rows[:12]:
        print(row)
else:
    print('missing')

for figure in [PERTURBATION_GRID_FIGURE, SALIENCY_COMPARISON_FIGURE, *[
    SALIENCY_COMPARISON_FIGURE.with_name(f'{SALIENCY_COMPARISON_FIGURE.stem}_{method}{SALIENCY_COMPARISON_FIGURE.suffix}')
    for method in XAI_METHODS
]]:
    print(figure)
    if figure.exists():
        display(Image(filename=str(figure)))


### Saliency Metric Interpretation

The table should be read by comparing methods under the same perturbation. Large prediction-change rate indicates model instability. Low IoU and low Spearman indicate explanation instability.

A useful comparison is the pairwise logic of the methods:

- If **Grad-CAM** and **Score-CAM** disagree strongly, the localization depends on whether channels are weighted by gradients or by masked-input scores.
- If **Integrated Gradients** and **Expected Gradients** disagree strongly, the explanation depends on the baseline choice and reference distribution.
- If all methods drift under background perturbation, the result supports the shortcut-dependence hypothesis more strongly than any single heatmap.


In [ ]:
if STRESS_METRICS_CSV.exists():
    metrics = pd.read_csv(STRESS_METRICS_CSV)
    iou_column = next((column for column in metrics if column.startswith('iou_top_')), None)
    if iou_column is None:
        raise ValueError('Stress metrics CSV is missing an iou_top_* column.')
    required_confidence_columns = {'original_target_probability', 'perturbed_target_probability', 'confidence_delta', 'confidence_drop'}
    missing_confidence_columns = required_confidence_columns.difference(metrics.columns)
    if missing_confidence_columns:
        raise ValueError(f'Regenerate the stress metrics CSV; missing corrected confidence columns: {sorted(missing_confidence_columns)}')
    for column in ['original_confidence', 'perturbed_confidence', *sorted(required_confidence_columns), iou_column, 'spearman']:
        metrics[column] = pd.to_numeric(metrics[column], errors='coerce')
    metrics['prediction_changed'] = metrics['prediction_changed'].astype(str).str.lower().eq('true')
    metrics['saliency_drift'] = (1.0 - metrics[iou_column]) + (1.0 - metrics['spearman']) / 2.0
    summary = (
        metrics.groupby(['xai_method', 'perturbation'], as_index=False)
        .agg(
            examples=('index', 'count'),
            prediction_change_rate=('prediction_changed', 'mean'),
            mean_confidence_drop=('confidence_drop', 'mean'),
            mean_iou=(iou_column, 'mean'),
            mean_spearman=('spearman', 'mean'),
            mean_saliency_drift=('saliency_drift', 'mean'),
        )
        .sort_values(['mean_saliency_drift', 'prediction_change_rate'], ascending=False)
    )
    print('Most unstable setting:')
    print(summary.iloc[0].to_dict())
    display(summary.round(3))
else:
    print('missing:', STRESS_METRICS_CSV)


## 2b. Advanced Attribution Audit

This section runs the same four attribution methods through faithfulness, region-allocation, sensitivity and class-discriminativeness diagnostics. It gives a stronger comparison than visual overlays alone.


In [ ]:
ADVANCED_REPORT = PROJECT_ROOT / 'outputs' / 'reports' / 'advanced_attribution_audit_notebook.csv'
ADVANCED_SUMMARY = PROJECT_ROOT / 'outputs' / 'reports' / 'advanced_attribution_audit_notebook_summary.csv'
ADVANCED_FIGURE_DIR = PROJECT_ROOT / 'outputs' / 'figures' / 'advanced_attribution_audit_notebook'

if RUN_ADVANCED_AUDIT:
    cmd = [
        sys.executable,
        str(PROJECT_ROOT / 'scripts' / 'audits' / 'run_advanced_attribution_audit.py'),
        '--manifest', str(MANIFEST),
        '--checkpoint', str(CHECKPOINT),
        '--methods', *XAI_METHODS,
        '--num-examples', '4',
        '--batch-size', '16',
        '--ig-steps', '16',
        '--ig-internal-batch-size', '4',
        '--expected-gradient-samples', '12',
        '--expected-gradient-baselines', '16',
        '--scorecam-max-channels', '48',
        '--scorecam-batch-size', '16',
        '--curve-steps', '10',
        '--report-output', str(ADVANCED_REPORT),
        '--summary-output', str(ADVANCED_SUMMARY),
        '--figure-dir', str(ADVANCED_FIGURE_DIR),
    ]
    print(' '.join(cmd))
    subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)
else:
    print('advanced attribution audit skipped; set RUN_ADVANCED_AUDIT=True to recompute it')

print('', ADVANCED_SUMMARY)
if ADVANCED_SUMMARY.exists():
    with ADVANCED_SUMMARY.open('r', newline='', encoding='utf-8') as handle:
        rows = list(csv.DictReader(handle))
    rows = [row for row in rows if row['method'] in XAI_METHODS]
    for row in rows:
        print({
            'method': row['method'],
            'examples': row['examples'],
            'faithfulness_gap': row['mean_faithfulness_gap'],
            'animal_saliency_ratio': row['mean_animal_saliency_ratio'],
            'sensitivity_iou_top20': row['mean_sensitivity_iou_top20'],
            'class_discriminativeness_iou_top20': row['mean_class_discriminativeness_iou_top20'],
        })
else:
    print('missing')

for method in XAI_METHODS:
    for suffix in ['deletion_insertion', 'class_discriminativeness']:
        figure = ADVANCED_FIGURE_DIR / f'{method}_{suffix}.png'
        print(figure)
        if figure.exists():
            display(Image(filename=str(figure)))


### Interpretation checkpoint

The notebook now recomputes the perturbation metrics for Grad-CAM, Score-CAM, Integrated Gradients and Expected Gradients. Read the stress-metrics table by comparing methods under the same perturbation: Grad-CAM and Score-CAM form the activation-map pair, while Integrated Gradients and Expected Gradients form the path-attribution pair.

Large prediction-change rates indicate model instability. Low IoU, low Spearman correlation and high saliency drift indicate explanation instability. If all four methods drift under background perturbations, the shortcut-dependence claim is stronger than any single heatmap.


## 3. Concept Transitions

Pixel-level maps identify regions of sensitivity but do not provide semantic structure. AwA2 class-level attributes allow transitions between original and perturbed predictions to be represented as changes in concept space.

For an antelope-to-walrus transition, the analysis compares the source and target concept vectors and reports gained, lost and shared attributes. This converts a class flip into a structured semantic error profile.


In [ ]:
if RUN_CONCEPTS:
    cmd = [
        sys.executable,
        str(PROJECT_ROOT / 'scripts' / 'experiments' / 'analyze_concept_profiles.py'),
        '--manifest', str(MANIFEST),
        '--metadata-root', str(METADATA_ROOT),
        '--stress-csv', str(PROJECT_ROOT / 'outputs' / 'reports' / 'phase5_saliency_metrics_notebook.csv'),
        '--class-profile-output', str(PROJECT_ROOT / 'outputs' / 'reports' / 'phase6_class_concepts_notebook.csv'),
        '--transition-output', str(PROJECT_ROOT / 'outputs' / 'reports' / 'phase6_concept_transitions_notebook.csv'),
        '--heatmap-output', str(PROJECT_ROOT / 'outputs' / 'figures' / 'phase6_class_concept_heatmap_notebook.png'),
        '--transition-figure-output', str(PROJECT_ROOT / 'outputs' / 'figures' / 'phase6_concept_transition_examples_notebook.png'),
    ]
    print(' '.join(cmd))
    subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)
else:
    print('Concept profile analysis skipped')

for figure in [
    PROJECT_ROOT / 'outputs' / 'figures' / 'phase6_class_concept_heatmap_notebook.png',
    PROJECT_ROOT / 'outputs' / 'figures' / 'phase6_concept_transition_examples_notebook.png',
]:
    print(figure)
    if figure.exists():
        display(Image(filename=str(figure)))


## 4. TCAV

TCAV means **Testing with Concept Activation Vectors**.

Procedure:

1. Select positive and negative examples for a concept using AwA2 attributes.
2. Extract internal ResNet activations at a chosen layer.
3. Train a linear separator between positive and negative activations.
4. Use the separator normal vector as the concept direction `v_c`.
5. Compute the directional derivative of a class score along that concept direction.

For class score `S_k` and activation vector `h_l(x)` at layer `l`:

```text
DirectionalDerivative(x, k, c) = grad_h S_k(h_l(x)) dot v_c
TCAV(k, c) = fraction of examples with DirectionalDerivative(x, k, c) > 0
```

A high TCAV score indicates that increasing the concept direction tends to increase the target class score at the selected layer. It is a sensitivity measure in representation space, not direct causal evidence.


In [ ]:
if RUN_TCAV:
    cmd = [
        sys.executable,
        str(PROJECT_ROOT / 'scripts' / 'experiments' / 'run_tcav.py'),
        '--manifest', str(MANIFEST),
        '--metadata-root', str(METADATA_ROOT),
        '--checkpoint', str(CHECKPOINT),
        '--score-output', str(PROJECT_ROOT / 'outputs' / 'reports' / 'phase7_tcav_scores_notebook.csv'),
        '--cav-output', str(PROJECT_ROOT / 'outputs' / 'reports' / 'phase7_cav_summary_notebook.csv'),
        '--heatmap-output', str(PROJECT_ROOT / 'outputs' / 'figures' / 'phase7_tcav_heatmap_notebook.png'),
        '--bar-output', str(PROJECT_ROOT / 'outputs' / 'figures' / 'phase7_tcav_top_scores_notebook.png'),
    ]
    print(' '.join(cmd))
    subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)
else:
    print('TCAV generation skipped')

for csv_path in [
    PROJECT_ROOT / 'outputs' / 'reports' / 'phase7_tcav_scores_notebook.csv',
    PROJECT_ROOT / 'outputs' / 'reports' / 'phase7_cav_summary_notebook.csv',
]:
    print('\n', csv_path)
    if csv_path.exists():
        with csv_path.open('r', newline='', encoding='utf-8') as handle:
            rows = list(csv.DictReader(handle))
        print('rows:', len(rows))
        for row in rows[:8]:
            print(row)

for figure in [
    PROJECT_ROOT / 'outputs' / 'figures' / 'phase7_tcav_heatmap_notebook.png',
    PROJECT_ROOT / 'outputs' / 'figures' / 'phase7_tcav_top_scores_notebook.png',
]:
    print(figure)
    if figure.exists():
        display(Image(filename=str(figure)))


### Interpreting TCAV Results

TCAV should be interpreted by separating semantically plausible associations from suspicious correlations. High `stripes -> zebra/tiger` is expected. High scores for unrelated pairs, such as aquatic concepts affecting non-aquatic animals, indicate that the learned concept direction may encode correlated dataset structure rather than a pure semantic attribute.


In [ ]:
tcav_path = PROJECT_ROOT / 'outputs' / 'reports' / 'phase7_tcav_scores_notebook.csv'
if tcav_path.exists():
    with tcav_path.open('r', newline='', encoding='utf-8') as handle:
        rows = list(csv.DictReader(handle))
    rows_sorted = sorted(rows, key=lambda r: float(r['tcav_score']), reverse=True)
    print('Highest TCAV sensitivities')
    for row in rows_sorted[:8]:
        print(f"{row['concept']} -> {row['target_class']}: score={float(row['tcav_score']):.3f}, n={row['n_eval']}")
    print('\nLowest TCAV sensitivities')
    for row in rows_sorted[-8:]:
        print(f"{row['concept']} -> {row['target_class']}: score={float(row['tcav_score']):.3f}, n={row['n_eval']}")
else:
    print('missing:', tcav_path)


### Technical conclusion

This notebook demonstrates that visually plausible explanations can be unstable under background perturbations. The concept analysis and TCAV sections provide a higher-level diagnostic layer, but they remain audit tools rather than causal proof. Suspicious concept-class associations require additional validation.
